## Imports


In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import pyarrow as pa
import pyarrow.feather as feather
from datetime import timezone, timedelta, datetime
import os


## Data Load and Preprocessing

Loading and refining the FAA wildlife strike dataset by standardizing headers, normalizing missing values to NaN, and pruning incomplete records. All local timestamps are synchronized into a unified UTC Unix format to ensure consistency across the analysis.

In [5]:
# -----------------------------
# DATA LOAD AND PREPROCESS
# -----------------------------

# Initial DataFrame load
df = pd.read_csv(r"/home/leonmortari/code/phd-ita/BS_Code/Public.csv")

# 1. COLUMN NAME CLEANUP
df.columns = df.columns.str.strip()

# 2. TRIM WHITESPACE FROM ALL TEXT COLUMNS (object dtype)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

# 3. UNKNOWN VALUE NORMALIZATION TO np.nan
unknown_values = ["Unknown", "UNKNOWN", "unk", "NaN", "nan", "None"]
# NOTE: str.strip() already handles "" and " "
df = df.replace(unknown_values, np.nan)

# 4. CLEAN REMAINING EMPTY STRINGS
df = df.replace("", np.nan)

# 5. DROP ROWS WHERE CRITICAL COLUMNS ARE NaN
df = df.dropna(subset=['TIME', 'INCIDENT_DATE'])

# 6. FINAL TIMESTAMP BUILD (preserving the original concatenation logic)
df["INCIDENT_DATE"] = pd.to_datetime(df["INCIDENT_DATE"])
df['TIMESTAMP_STR'] = df['INCIDENT_DATE'].astype(str) + ' ' + (df['TIME'] + ':00').astype(str)
df['DATETIME_UTC'] = pd.to_datetime(df['TIMESTAMP_STR']).dt.tz_localize(timezone.utc)

# Convert to numeric UNIX epoch seconds
df['TIMESTAMP_UNIX_SEC'] = (df['DATETIME_UTC'].astype(np.int64) // 10**9)

print("\nConverted values:")
print(df[['TIMESTAMP_STR', 'DATETIME_UTC', 'TIMESTAMP_UNIX_SEC']].head())

# Keep only target columns
df = df[[
    "DATETIME_UTC", "TIMESTAMP_UNIX_SEC", "INCIDENT_DATE", "TIME", "TIME_OF_DAY", "AIRPORT_ID",
    "AIRPORT", "RUNWAY", "AIRCRAFT", "PHASE_OF_FLIGHT",
    "HEIGHT", "SKY", "OPERATOR", "OPID", "REG", "FLT",
    "SPECIES", "SIZE"
]]

print("Data types per column:")
print(df.dtypes)

/tmp/ipykernel_7773/2562908417.py:6: DtypeWarning: Columns (8,9,17,20,80,91) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"/home/leonmortari/code/phd-ita/BS_Code/Public.csv")



Converted values:
           TIMESTAMP_STR              DATETIME_UTC  TIMESTAMP_UNIX_SEC
9    1990-08-07 20:25:00 1990-08-07 20:25:00+00:00           650060700
43   1990-08-05 19:05:00 1990-08-05 19:05:00+00:00           649883100
117  1991-04-17 13:15:00 1991-04-17 13:15:00+00:00           671894100
131  1990-10-15 10:07:00 1990-10-15 10:07:00+00:00           655985220
139  1990-09-21 10:00:00 1990-09-21 10:00:00+00:00           653911200
Data types per column:
DATETIME_UTC          datetime64[ns, UTC]
TIMESTAMP_UNIX_SEC                  int64
INCIDENT_DATE              datetime64[ns]
TIME                               object
TIME_OF_DAY                        object
AIRPORT_ID                         object
AIRPORT                            object
RUNWAY                             object
AIRCRAFT                           object
PHASE_OF_FLIGHT                    object
HEIGHT                            float64
SKY                                object
OPERATOR                    

## Season and Time-of-Day Determination

Implementing custom classification for Northern Hemisphere seasons and diurnal periods (Night, Dawn, Day, Dusk). This stage maps incident dates and hours to their respective ecological contexts while ensuring handling of missing values and standardized time formats

In [8]:
# -----------------------------
# SEASON DETERMINATION (NORTHERN HEMISPHERE)
# -----------------------------
def get_season(date):
    if pd.isna(date):
        return None

    month = date.month
    day = date.day

    if (month == 6 and day >= 21) or (month in [7, 8]) or (month == 9 and day < 23):
        return "Summer"
    elif (month == 9 and day >= 23) or (month in [10, 11]) or (month == 12 and day < 22):
        return "Fall"
    elif (month == 12 and day >= 22) or (month in [1, 2]) or (month == 3 and day < 20):
        return "Winter"
    elif (month == 3 and day >= 20) or (month in [4, 5]) or (month == 6 and day < 21):
        return "Spring"
    return None

df["SEASON"] = df["INCIDENT_DATE"].apply(get_season)

# Time-of-day classification
def derive_time_of_day(hour):
    if pd.isna(hour):
        return None
    h = int(hour)
    if 0 <= h <= 5: return "Night"
    if 6 <= h <= 7: return "Dawn"
    if 8 <= h <= 17: return "Day"
    if 18 <= h <= 19: return "Dusk"
    if 20 <= h <= 23: return "Night"
    return None

time_not_na = ~df["TIME"].isna()
df.loc[time_not_na, 'TEMP_HOUR'] = pd.to_datetime(
    df.loc[time_not_na, 'TIME'].astype(str),
    format='%H:%M',
    errors='coerce'
).dt.hour

time_is_nan = df['TEMP_HOUR'].isna() & time_not_na
df.loc[time_is_nan, 'TEMP_HOUR'] = df.loc[time_is_nan, 'TIME'].apply(
    lambda x: int(x) if isinstance(x, (float, int)) else None
)

print(f"Not NaN count {(~df.get('TIME_OF_DAY', pd.Series()).isna()).sum()} before")
df.loc[time_not_na, 'TIME_OF_DAY'] = df.loc[time_not_na, 'TEMP_HOUR'].apply(derive_time_of_day)
print(f"Not NaN count {(~df['TIME_OF_DAY'].isna()).sum()} after")

df = df.drop(columns=['TEMP_HOUR'], errors='ignore')
df.head()

Not NaN count 156059 before
Not NaN count 194299 after


/tmp/ipykernel_7773/2925577880.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[time_is_nan, 'TEMP_HOUR'] = df.loc[time_is_nan, 'TIME'].apply(


,DATETIME_UTC,TIMESTAMP_UNIX_SEC,INCIDENT_DATE,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,RUNWAY,AIRCRAFT,PHASE_OF_FLIGHT,HEIGHT,SKY,OPERATOR,OPID,REG,FLT,SPECIES,SIZE,SEASON
9,1990-08-07 20:25:00+00:00,650060700,1990-08-07,20:25,Night,KSTL,LAMBERT-ST LOUIS INTL,NaN,C-152,Climb,200.0,No Cloud,PRIVATELY OWNED,PVT,NaN,NaN,Unknown bird - large,Large,Summer
43,1990-08-05 19:05:00+00:00,649883100,1990-08-05,19:05,Dusk,KBWI,BALTIMORE/WASH INTL THURGOOD MARSHAL ARPT,NaN,FOKKER F28 MK 1000,Landing Roll,0.0,Some Cloud,1US AIRWAYS,USA,NaN,NaN,Unknown bird - small,Small,Summer
117,1991-04-17 13:15:00+00:00,671894100,1991-04-17,13:15,Day,KDCA,RONALD REAGAN WASHINGTON NATIONAL ARPT,NaN,B-727,Climb,100.0,No Cloud,AMERICAN AIRLINES,AAL,NaN,NaN,Ring-billed gull,Medium,Spring
131,1990-10-15 10:07:00+00:00,655985220,1990-10-15,10:07,Day,KTRI,TRI-CITIES REGIONAL TN/VA ARPT,23,MERLIN IV,Approach,1500.0,No Cloud,NASHVILLE EAGLE,NAE,NaN,NaN,Unknown bird - medium,Medium,Fall
139,1990-09-21 10:00:00+00:00,653911200,1990-09-21,10:00,Day,KMWH,GRANT COUNTY ARPT,32,B-747-1/200,Climb,3000.0,No Cloud,JAPAN AIRLINES,JAL,JA8104,NaN,Swallows,Small,Summer


## Data Cleaning: Missing Value Normalization

Standardizing null values and filtering out incomplete records based on mandatory flight and temporal fields.

In [10]:
# -----------------------------
# DATA CLEANING: MISSING VALUE NORMALIZATION
# -----------------------------
unknown_values = ["Unknown", "UNKNOWN", "unk", "NaN", "nan", "None", "", " "]
df = df.replace(unknown_values, np.nan)

df = df.dropna(subset=["INCIDENT_DATE", "TIME", "FLT", "OPID"])

data_strike = df[[
    "DATETIME_UTC", "TIMESTAMP_UNIX_SEC", "INCIDENT_DATE", "SEASON", "TIME", "TIME_OF_DAY", "AIRPORT_ID",
    "AIRPORT", "RUNWAY", "AIRCRAFT", "PHASE_OF_FLIGHT",
    "HEIGHT", "SKY", "OPERATOR", "OPID", "REG", "FLT", "SPECIES", "SIZE"
]].copy()

data_strike = data_strike.reset_index(drop=True)
data_strike

,DATETIME_UTC,TIMESTAMP_UNIX_SEC,INCIDENT_DATE,SEASON,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,RUNWAY,AIRCRAFT,PHASE_OF_FLIGHT,HEIGHT,SKY,OPERATOR,OPID,REG,FLT,SPECIES,SIZE
0,1997-11-22 15:20:00+00:00,880212000,1997-11-22,Fall,15:20,Day,KMCO,ORLANDO INTL,17R,B-737-200,Landing Roll,0.0,NaN,DELTA AIR LINES,DAL,NaN,2437,Bald eagle,Large
1,1997-12-10 06:45:00+00:00,881736300,1997-12-10,Fall,06:45,Dawn,KCLT,CHARLOTTE/DOUGLAS INTL ARPT,18R,DC-9-50,Take-off Run,0.0,Overcast,NORTHWEST AIRLINES,NWA,N781NC,1579,Mourning dove,Small
2,1998-10-09 20:35:00+00:00,907965300,1998-10-09,Fall,20:35,Night,KPIT,PITTSBURGH INTL ARPT,28,DC-9-30,Take-off Run,0.0,Overcast,1US AIRWAYS,USA,N955VJ,426,Owls,Medium
3,1999-02-03 10:45:00+00:00,918038700,1999-02-03,Winter,10:45,Day,KSLC,SALT LAKE CITY INTL,17,B-737-300,Take-off Run,0.0,No Cloud,SOUTHWEST AIRLINES,SWA,N350SW,1468,Horned lark,Small
4,1997-09-27 11:00:00+00:00,875358000,1997-09-27,Fall,11:00,Day,KJFK,JOHN F KENNEDY INTL,4R,B-767-200,Landing Roll,0.0,No Cloud,AMERICAN AIRLINES,AAL,N323AA,115,Tree swallow,Small
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130397,2025-10-13 23:00:00+00:00,1760396400,2025-10-13,Fall,23:00,Night,KMLI,QUAD CITY ARPT,09/27,EMB-170,Approach,4000.0,Some Cloud,ENVOY AIR,ENY,N230NN,4034,Hermit thrush,Small
130398,2025-10-13 19:27:00+00:00,1760383620,2025-10-13,Fall,19:27,Dusk,KSLC,SALT LAKE CITY INTL,16L,A-321,Approach,5000.0,Some Cloud,DELTA AIR LINES,DAL,N592DT,887,Brazilian free-tailed bat,Small
130399,2024-12-25 15:10:00+00:00,1735139400,2024-12-25,Winter,15:10,Day,KEWR,NEWARK LIBERTY INTL ARPT,04R,A-321,Landing Roll,0.0,Some Cloud,TAP AIR PORTUGAL,TAP,CS-TXL,201,Horned lark,Small
130400,2025-01-30 15:38:00+00:00,1738251480,2025-01-30,Winter,15:38,Day,KEWR,NEWARK LIBERTY INTL ARPT,22R,B-757-200,Climb,NaN,No Cloud,UNITED AIRLINES,UAL,N58101,2852,Horned lark,Small


## Data Unification

Standardizing date formats and sanitizing operational metadata (Registration/Operator ID) for uniformity. This stage handles the type conversion of flight numbers and constructs the CALLSIGN column, facilitating precise tracking and identification of specific flight operations within the strike database.

In [11]:
# -----------------------------
# DATA UNIFICATION
# -----------------------------
data_strike.loc[:, "INCIDENT_DATE"] = pd.to_datetime(data_strike["INCIDENT_DATE"]).dt.strftime("%Y-%m-%d")
data_strike.loc[:, "REG"] = data_strike["REG"].astype(str).str.upper().str.strip()
data_strike.loc[:, "OPID"] = data_strike["OPID"].astype(str).str.strip()
data_strike.loc[:, "FLT"] = pd.to_numeric(data_strike["FLT"], errors='coerce').fillna(0).astype(int).astype(str).str.strip()
data_strike.loc[:, "CALLSIGN"] = data_strike["OPID"] + data_strike["FLT"]
data_strike.head(3)

,DATETIME_UTC,TIMESTAMP_UNIX_SEC,INCIDENT_DATE,SEASON,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,RUNWAY,AIRCRAFT,PHASE_OF_FLIGHT,HEIGHT,SKY,OPERATOR,OPID,REG,FLT,SPECIES,SIZE,CALLSIGN
0,1997-11-22 15:20:00+00:00,880212000,1997-11-22,Fall,15:20,Day,KMCO,ORLANDO INTL,17R,B-737-200,Landing Roll,0.0,NaN,DELTA AIR LINES,DAL,NAN,2437,Bald eagle,Large,DAL2437
1,1997-12-10 06:45:00+00:00,881736300,1997-12-10,Fall,06:45,Dawn,KCLT,CHARLOTTE/DOUGLAS INTL ARPT,18R,DC-9-50,Take-off Run,0.0,Overcast,NORTHWEST AIRLINES,NWA,N781NC,1579,Mourning dove,Small,NWA1579
2,1998-10-09 20:35:00+00:00,907965300,1998-10-09,Fall,20:35,Night,KPIT,PITTSBURGH INTL ARPT,28,DC-9-30,Take-off Run,0.0,Overcast,1US AIRWAYS,USA,N955VJ,426,Owls,Medium,USA426


## Species Classification

Mapping individual species to higher-level biological categories for streamlined analysis. This logic consolidates the dataset into functional groups and establishes an IS_BIRD indicator, enabling a clear distinction between avian data and other non-bird wildlife records.

In [13]:
# -----------------------------
# SPECIES CLASSIFICATION
# -----------------------------

species_mapping = {
    "Waterfowl (Ducks, Geese, Swans)": [
        "duck", "teal", "wigeon", "scoter", "eider", "goldeneye", "goose", "swan", "brant",
        "cackling goose", "pintail", "merganser", "mallard", "redhead", "northern shoveler",
        "mottled duck", "gadwall", "muscovy duck", "american wigeon", "american black duck",
        "canvasback", "greater white-fronted goose", "lesser scaup", "ring-necked duck",
        "common goldeneye", "long-tailed duck", "bufflehead", "surf scoter", "black-bellied whistling-duck",
        "barrow's goldeneye", "greater scaup", "cinnamon teal", "hawaiian duck", "diving duck (aythya)",
        "snow goose"
    ],
    "Raptor & Owl": [
        "hawk", "kestrel", "falcon", "vulture", "eagle", "osprey", "harrier", "merlin", "caracara",
        "buzzard", "owl", "screech-owl", "kite", "peregrine falcon", "red-tailed hawk",
        "american kestrel", "bald eagle", "turkey vulture", "great horned owl", "swainson's hawk",
        "red-shouldered hawk", "northern harrier", "cooper's hawk", "prairie falcon", "crested caracara",
        "northern hawk owl", "sharp-shinned hawk", "ferruginous hawk", "golden eagle", "mississippi kite",
        "harris's hawk", "eurasian buzzard", "great gray owl", "gyrfalcon"
    ],
    "Wader & Heron": [
        "sandpiper", "plover", "snipe", "dowitcher", "turnstone", "godwit", "stilt", "avocet",
        "phalarope", "curlew", "ibis", "spoonbill", "willet", "shorebird", "heron", "egret",
        "bittern", "cormorant", "pelican", "loon", "grebe", "oystercatcher", "stork", "gallinule",
        "crane", "killdeer", "pacific golden-plover", "western cattle egret", "american coot",
        "great blue heron", "sandhill crane", "black-bellied plover", "upland sandpiper",
        "western sandpiper", "american bittern", "california gull", "western gull",
        "american golden-plover", "black-crowned night heron", "least sandpiper",
        "double-crested cormorant", "semipalmated plover", "lesser yellowlegs", "snowy egret",
        "wilson's snipe", "dunlin", "semipalmated sandpiper", "great egret", "green heron",
        "wood stork", "sanderling", "long-billed dowitcher", "pectoral sandpiper", "pied-billed grebe",
        "common loon", "anhinga", "yellow-crowned night heron", "whooping crane", "whimbrel",
        "red knot", "short-billed dowitcher", "red-throated loon", "black-necked stilt",
        "greater yellowlegs", "american avocet", "spotted sandpiper", "least bittern", "sora",
        "wilson's plover", "piping plover", "red-necked phalarope", "wilson's phalarope",
        "long-billed curlew", "ruddy turnstone", "red-necked grebe",
        "brandt's cormorant", "white-faced ibis", "marbled godwit", "glossy ibis", "roseate spoonbill",
        "black skimmer", "tricolored heron", "forster's tern", "connecticut warbler", "eurasian collared dove"
    ],
    "Gull & Tern": [
        "gull", "tern", "noddy", "kittiwake", "ring-billed gull", "herring gull", "laughing gull",
        "short-billed gull", "heermann's gull", "bonaparte's gull", "caspian tern", "franklin's gull",
        "great black-backed gull", "glaucous-winged gull", "lesser black-backed gull", "common tern",
        "least tern", "sooty tern", "royal tern", "elegant tern"
    ],
    "Passerine & Small Bird": [
        "sparrow", "swallow", "robin", "starling", "lark", "kingfisher", "wren", "thrush", "pipit",
        "grosbeak", "bunting", "crow", "raven", "mockingbird", "grackle", "jay", "magpie",
        "oriole", "woodpecker", "flycatcher", "vireo", "tanager", "chat", "cardinal", "finch",
        "nuthatch", "hummingbird", "parakeet", "parrot", "canary", "drongo", "myna", "waxwing",
        "chickadee", "warbler", "swift", "western meadowlark", "american robin", "blackbird",
        "savannah sparrow", "american crow", "horned lark", "purple martin", "northern mockingbird",
        "snow bunting", "eastern meadowlark", "brown-headed cowbird", "bank swallow", "common yellowthroat",
        "common raven", "black-and-white warbler", "common grackle", "great-tailed grackle",
        "hairy woodpecker", "scissor-tailed flycatcher", "red-vented bulbul", "common myna",
        "fox sparrow", "american pipit", "acadian flycatcher", "song sparrow", "western bluebird",
        "american goldfinch", "yellow-billed magpie", "dark-eyed junco", "cedar waxwing",
        "western kingbird", "blue jay", "brewer's blackbird", "black-billed magpie", "eastern kingbird",
        "lark bunting", "house sparrow", "swainson's thrush", "varied thrush", "hermit thrush",
        "blackpoll warbler", "yellow warbler", "yellow-rumped warbler", "eastern phoebe",
        "lapland longspur", "pine warbler", "northern rough-winged swallow", "orchard oriole",
        "mountain chickadee", "chestnut-collared longspur", "ovenbird", "northern flicker",
        "brown thrasher", "ruby-throated hummingbird", "chipping sparrow", "golden-crowned kinglet",
        "grasshopper sparrow", "bohemian waxwing", "wilson's warbler", "loggerhead shrike",
        "american tree sparrow", "lincoln's sparrow", "field sparrow", "gray catbird",
        "blue-gray gnatcatcher", "vesper sparrow", "eurasian skylark", "white-throated swift",
        "anna's hummingbird", "pine siskin", "western tanager", "bell's sparrow", "mountain bluebird",
        "boat-tailed grackle", "northern parula", "bobolink", "california towhee", "cactus wren",
        "tropical mockingbird", "cave swallow", "ruby-crowned kinglet", "broad-winged hawk",
        "carolina chickadee", "swamp sparrow", "northern house wren", "white-crowned sparrow",
        "indian silverbill", "lesser goldfinch", "western flycatcher", "lark sparrow", "veery",
        "cassin's vireo", "yellow-headed blackbird", "yellow-billed cuckoo", "gray-cheeked thrush",
        "purple finch", "nelson's sparrow", "japanese white-eye", "tennessee warbler",
        "blackburnian warbler", "orange-crowned warbler", "baltimore oriole", "black drongo",
        "bushtit", "black-throated blue warbler", "red-eyed vireo", "canada warbler", "pine grosbeak",
        "black-throated green warbler", "yellow-breasted chat", "warbling vireo", "magnolia warbler",
        "northern waterthrush", "brewer's sparrow", "black-headed grosbeak", "eastern whip-poor-will",
        "dickcissel", "willow flycatcher", "rose-breasted grosbeak", "yellow-throated warbler",
        "american redstart", "cape may warbler", "marsh wren", "sprague's pipit", "indigo bunting",
        "palm warbler", "harris's sparrow", "prairie warbler", "eastern bluebird", "green-tailed towhee",
        "mourning warbler", "black-billed cuckoo", "kentucky warbler", "smith's longspur",
        "carolina wren", "eastern wood-pewee", "yellow-bellied flycatcher", "summer tanager",
        "scarlet tanager", "chestnut-sided warbler", "clay-colored sparrow", "wood thrush",
        "hooded warbler", "leconte's sparrow", "least flycatcher", "hammond's flycatcher",
        "eastern towhee", "winter wren", "bay-breasted warbler", "redpoll", "meadow pipit",
        "say's phoebe", "red crossbill", "worm-eating warbler", "nanday parakeet",
        "yellow-throated vireo", "white-eyed vireo", "hooded oriole", "black-throated gray warbler",
        "bullock's oriole", "ash-throated flycatcher", "downy woodpecker", "black-chinned hummingbird",
        "sedge wren", "painted bunting", "hermit warbler", "saffron finch", "common raven"
    ],
    "Pigeon & Dove": ["pigeon", "dove", "rock pigeon", "mourning dove", "zebra dove", "spotted dove", "white-winged dove", "common ground dove", "band-tailed pigeon", "eurasian collared dove", "zenaida dove", "white-crowned pigeon"],
    "Upland Gamebird": ["pheasant", "quail", "grouse", "francolin", "partridge", "chukar", "wild turkey", "northern bobwhite", "gray partridge", "greater sage-grouse", "sharp-tailed grouse", "scaled quail", "gambel's quail", "helmeted guineafowl"],
    "Mammal": [
        "deer", "coyote", "rabbit", "bat", "opossum", "skunk", "racoon", "porcupine", "squirrel",
        "mink", "badger", "nutria", "canid", "bear", "pig", "moose", "elk", "caribou", "fox",
        "coypu", "cattle", "otter", "woodchuck", "pronghorn", "armadillo", "gopher", "dog", "cat",
        "peccary", "mongoose", "marmot", "prairie dog", "muskrat", "hare", "white-tailed deer",
        "river otter", "wapiti (elk)", "eastern cottontail", "mule deer", "black-tailed jackrabbit",
        "striped skunk", "nine-banded armadillo", "white-tailed jackrabbit", "gunnison's prairie dog",
        "north american porcupine", "yellow-bellied marmot", "white-tailed prairie dog",
        "black-tailed prairie dog", "eastern red bat", "collared peccary", "small indian mongoose",
        "kit fox", "common gray fox", "big brown bat", "silver-haired bat", "hoary bat",
        "florida bonneted bat", "yuma myotis", "jamaican fruit bat", "evening bat", "american badger",
        "american mink"
    ],
    "Reptile": ["turtle", "snake", "alligator", "lizard", "iguana", "terrapin", "rattlesnake", "common snapping turtle", "pond slider", "diamondback terrapin", "gopher tortoise", "eastern diamondback rattlesnake", "green iguana"],
    "Unknown": ["unknown bird - medium", "unknown bird - small", "unknown bird - large"]
}

def consolidate_species(species):
    species_lower = str(species).lower()
    for group, keywords in species_mapping.items():
        if any(keyword in species_lower for keyword in keywords):
            return group
    return "Other"

data_strike['SPECIES_GROUP'] = data_strike['SPECIES'].apply(consolidate_species)

def is_bird_species(species_group):
    non_bird_groups = ['Mammal', 'Reptile']
    return False if species_group in non_bird_groups else True

data_strike['SPECIES_GROUP'] = data_strike['SPECIES'].apply(consolidate_species)
data_strike['IS_BIRD'] = data_strike['SPECIES_GROUP'].apply(is_bird_species)
data_strike

# -----------------------------
# BIRD-ONLY ANALYSIS
# -----------------------------
data_strike = data_strike[data_strike['IS_BIRD'] == True].copy()
data_strike

,DATETIME_UTC,TIMESTAMP_UNIX_SEC,INCIDENT_DATE,SEASON,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,RUNWAY,AIRCRAFT,...,SKY,OPERATOR,OPID,REG,FLT,SPECIES,SIZE,CALLSIGN,SPECIES_GROUP,IS_BIRD
0,1997-11-22 15:20:00+00:00,880212000,1997-11-22,Fall,15:20,Day,KMCO,ORLANDO INTL,17R,B-737-200,...,NaN,DELTA AIR LINES,DAL,NAN,2437,Bald eagle,Large,DAL2437,Raptor & Owl,True
1,1997-12-10 06:45:00+00:00,881736300,1997-12-10,Fall,06:45,Dawn,KCLT,CHARLOTTE/DOUGLAS INTL ARPT,18R,DC-9-50,...,Overcast,NORTHWEST AIRLINES,NWA,N781NC,1579,Mourning dove,Small,NWA1579,Pigeon & Dove,True
2,1998-10-09 20:35:00+00:00,907965300,1998-10-09,Fall,20:35,Night,KPIT,PITTSBURGH INTL ARPT,28,DC-9-30,...,Overcast,1US AIRWAYS,USA,N955VJ,426,Owls,Medium,USA426,Raptor & Owl,True
3,1999-02-03 10:45:00+00:00,918038700,1999-02-03,Winter,10:45,Day,KSLC,SALT LAKE CITY INTL,17,B-737-300,...,No Cloud,SOUTHWEST AIRLINES,SWA,N350SW,1468,Horned lark,Small,SWA1468,Passerine & Small Bird,True
4,1997-09-27 11:00:00+00:00,875358000,1997-09-27,Fall,11:00,Day,KJFK,JOHN F KENNEDY INTL,4R,B-767-200,...,No Cloud,AMERICAN AIRLINES,AAL,N323AA,115,Tree swallow,Small,AAL115,Passerine & Small Bird,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130396,2025-10-13 15:57:00+00:00,1760371020,2025-10-13,Fall,15:57,Day,KOTH,SOUTHWEST OREGON REGIONAL ARPT,5,NaN,...,NaN,VISTAJET,VJT,NAN,505,Unknown bird,NaN,VJT505,Other,True
130397,2025-10-13 23:00:00+00:00,1760396400,2025-10-13,Fall,23:00,Night,KMLI,QUAD CITY ARPT,09/27,EMB-170,...,Some Cloud,ENVOY AIR,ENY,N230NN,4034,Hermit thrush,Small,ENY4034,Passerine & Small Bird,True
130399,2024-12-25 15:10:00+00:00,1735139400,2024-12-25,Winter,15:10,Day,KEWR,NEWARK LIBERTY INTL ARPT,04R,A-321,...,Some Cloud,TAP AIR PORTUGAL,TAP,CS-TXL,201,Horned lark,Small,TAP201,Passerine & Small Bird,True
130400,2025-01-30 15:38:00+00:00,1738251480,2025-01-30,Winter,15:38,Day,KEWR,NEWARK LIBERTY INTL ARPT,22R,B-757-200,...,No Cloud,UNITED AIRLINES,UAL,N58101,2852,Horned lark,Small,UAL2852,Passerine & Small Bird,True


## Save Processed Data to IPC Format

In [15]:


OUTPUT_DIR = "/home/leonmortari/code/phd-ita/BS_Code/Analise"
os.makedirs(OUTPUT_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
FILENAME = f"data_strike_z_{timestamp}.ipc"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, FILENAME)

def save_optimized_ipc(df: pd.DataFrame, path: str):
    """
    Save Pandas DataFrame to IPC (Feather) via Polars with light compression.
    """
    if df.empty:
        print("⚠️ The DataFrame is empty. Nothing to save.")
        return

    try:
        for col in df.select_dtypes(include=["datetimetz"]).columns:
            df[col] = df[col].dt.tz_convert("UTC")

        df_polars = pl.from_pandas(df, include_index=False)
        df_polars.write_ipc(path, compression="lz4")

        print(f"\n✅ DataFrame successfully saved in IPC (Feather) format.")
        print(f"📁 Path: {path}")
        print(f"🧾 Rows: {df_polars.height:,} | Columns: {df_polars.width}")
        print(f"🕒 Main dtypes: {df_polars.dtypes[:5]} ...")

    except Exception as e:
        print(f"❌ ERROR saving IPC file: {e}")

save_optimized_ipc(data_strike, OUTPUT_PATH)


✅ DataFrame successfully saved in IPC (Feather) format.
📁 Path: /home/leonmortari/code/phd-ita/BS_Code/Analise/data_strike_z_20260328_060328.ipc
🧾 Rows: 127,236 | Columns: 22
🕒 Main dtypes: [Datetime(time_unit='ns', time_zone='UTC'), Int64, Datetime(time_unit='ns', time_zone=None), String, String] ...
